# 03 — Modification: Alternative Typicality Measures

**Hypothesis:** Since SimCLR features are L2-normalised, cosine distance or density-aware measures (LOF, KDE) may better capture semantic typicality than Euclidean distance.

**Controlled experiment:** Only the typicality function changes; features, clustering, and classifier training are identical.

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, os.path.dirname(os.getcwd()))
else:
    sys.path.insert(0, os.getcwd())

import numpy as np
from torchvision import datasets
from src.active_learning import run_repeated_experiment
from src.augmentations import get_classifier_train_transform, get_classifier_test_transform
from src.utils import get_device, set_seed, MODELS_DIR, DATA_DIR, save_results, load_results
set_seed(42)
device = get_device()

In [ ]:
features = np.load(str(MODELS_DIR / 'simclr_features.npy'))
train_dataset = datasets.CIFAR10(root=str(DATA_DIR), train=True, download=True, transform=get_classifier_train_transform())
test_dataset = datasets.CIFAR10(root=str(DATA_DIR), train=False, download=True, transform=get_classifier_test_transform())

## 1. Run All Typicality Variants

In [ ]:
variants = ['cosine', 'lof', 'kde']  # euclidean already run in NB02
all_results = {}

for typ in variants:
    print(f'\nRunning: {typ}')
    res = run_repeated_experiment(
        features=features, train_dataset=train_dataset, test_dataset=test_dataset,
        strategy='typiclust', budget_per_round=10, n_rounds=5, n_reps=5,
        classifier_epochs=200, device=device, typicality_fn=typ, base_seed=42,
    )
    all_results[typ] = res
    save_results(res, f'typiclust_{typ}_b10.json')

# Load baseline results
all_results['euclidean'] = load_results('typiclust_euclidean_b10.json')
all_results['random'] = load_results('random_b10.json')

## 2. Comparison Plot

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5.5))
styles = {
    'euclidean': ('tab:blue', 'o', '-'),
    'cosine': ('tab:orange', '^', '-'),
    'lof': ('tab:green', 's', '-'),
    'kde': ('tab:red', 'D', '-'),
    'random': ('tab:gray', 'x', '--'),
}

for name, res in all_results.items():
    c, mk, ls = styles[name]
    b = res['cumulative_budget']
    m = np.array(res['mean_accuracy']); s = np.array(res['se_accuracy'])
    label = f'TypiClust ({name})' if name != 'random' else 'Random'
    ax.plot(b, m, f'{mk}{ls}', label=label, color=c, markersize=6)
    ax.fill_between(b, m-s, m+s, alpha=0.15, color=c)

ax.set_xlabel('Cumulative Budget'); ax.set_ylabel('Test Accuracy (%)')
ax.set_title('Typicality Measure Comparison on CIFAR-10')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('../figures/typicality_comparison.png', dpi=150); plt.show()

## 3. Summary Table

In [ ]:
import pandas as pd

rows = []
for name, res in all_results.items():
    for b, a, s in zip(res['cumulative_budget'], res['mean_accuracy'], res['se_accuracy']):
        rows.append({'Strategy': name, 'Budget': b, 'Acc': f'{a:.1f} +/- {s:.1f}'})
df = pd.DataFrame(rows)
print(df.pivot(index='Budget', columns='Strategy', values='Acc').to_string())

## 4. Statistical Tests

In [ ]:
from scipy import stats

print('Comparison at final budget (B=50):')
baseline = np.array(all_results['euclidean']['all_accuracies'])[:, -1]

for name in ['cosine', 'lof', 'kde']:
    variant = np.array(all_results[name]['all_accuracies'])[:, -1]
    t, p = stats.ttest_ind(variant, baseline)
    diff = variant.mean() - baseline.mean()
    print(f'  {name} vs euclidean: diff={diff:+.2f}%, t={t:.3f}, p={p:.4f}')
    print(f'    Note: only {len(baseline)} reps — limited statistical power')